In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler

# --- Reload raw data ---
expr = pd.read_csv('../data/raw/HiSeqV2', sep='\t', index_col=0)
mut = pd.read_csv('../data/raw/BRCA_mc3_gene_level.txt', sep='\t', index_col=0)
clin = pd.read_csv('../data/raw/TCGA.BRCA.sampleMap_BRCA_clinicalMatrix', sep='\t', index_col=0)

expr_samples = set(expr.columns)
mut_samples = set(mut.columns)
labeled_patients = set(clin[clin['PAM50Call_RNAseq'].notna()].index)
common_593 = set(sorted(expr_samples & mut_samples & labeled_patients))

# --- Find NEW patients: have expr + mut, but NOT in the 593 used ---
all_expr_mut = expr_samples & mut_samples
new_patients = sorted(all_expr_mut - common_593)
print(f"Found {len(new_patients)} new patients never used in training/testing")

# Some of these might still have a PAM50 label (just check)
new_with_label = [p for p in new_patients if p in labeled_patients]
print(f"Of these, {len(new_with_label)} have a known PAM50 label (for comparison)")

# --- Load metadata for feature columns + recreate scaler ---
with open('../data/processed/metadata.pkl', 'rb') as f:
    meta = pickle.load(f)

expr_features = meta['expr_features']
mut_features = meta['mut_features']

# Recreate scaler using the SAME 593 patients used for training (to match original transform)
expr_subset_593 = expr[sorted(common_593)].T
expr_filtered_593 = expr_subset_593[expr_features]
scaler = StandardScaler()
scaler.fit(expr_filtered_593)

# --- Export 3 new patients as CSVs ---
import os
os.makedirs('../data/sample_patients', exist_ok=True)

for p in new_patients[:3]:
    expr_row = expr[[p]].T[expr_features]
    expr_scaled = scaler.transform(expr_row)
    mut_row = mut[[p]].T[mut_features].values

    row = np.hstack([expr_scaled, mut_row])
    df = pd.DataFrame(row, columns=expr_features + mut_features)

    label_str = clin.loc[p, 'PAM50Call_RNAseq'] if p in new_with_label else "unknown"
    filename = f'../data/sample_patients/new_{p}_{label_str}.csv'
    df.to_csv(filename, index=False)
    print(f"Saved {filename}")

Found 196 new patients never used in training/testing
Of these, 0 have a known PAM50 label (for comparison)
Saved ../data/sample_patients/new_TCGA-3C-AAAU-01_unknown.csv
Saved ../data/sample_patients/new_TCGA-3C-AALI-01_unknown.csv
Saved ../data/sample_patients/new_TCGA-3C-AALJ-01_unknown.csv
